# Heart Disease Detection - Training Pipeline
## Classification using Decision Tree, Random Forest, Logistic Regression & SVM

### Step 1: Load Dataset

In [ ]:
# Upload heart.csv to Colab or mount Google Drive
# Dataset columns: age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target

import pandas as pd
import numpy as np

# Option 1: If uploaded to Colab
# df = pd.read_csv('heart.csv')

# Option 2: From Google Drive (uncomment after mounting)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/heart.csv')

# Option 3: Use UCI dataset URL if no local file
try:
    df = pd.read_csv('heart.csv')
except:
    url = 'https://raw.githubusercontent.com/plotly/datasets/master/heart.csv'
    df = pd.read_csv(url)

df.head()

### Step 2: Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure target column exists (some datasets use 'num' or 'target')
if 'target' not in df.columns and 'num' in df.columns:
    df['target'] = (df['num'] > 0).astype(int)
elif 'target' not in df.columns:
    df['target'] = df.iloc[:, -1]  # Last column as target

# Target variable analysis - Bar chart
plt.figure(figsize=(6, 4))
df['target'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Target Variable Distribution (0=No Disease, 1=Heart Disease)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Target distribution:")
print(df['target'].value_counts())

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots/Histograms of numerical features
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    if col in df.columns:
        axes[i].boxplot(df[col].dropna())
        axes[i].set_title(col)

axes[-1].axis('off')
plt.suptitle('Boxplots of Numerical Features')
plt.tight_layout()
plt.show()

### Step 3: Data Preprocessing & Feature Engineering

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Handle missing values (none found in UCI dataset)
print("Missing values:", df.isnull().sum().sum())
df = df.dropna()

# Define features and target
target_col = 'target' if 'target' in df.columns else df.columns[-1]
y = df[target_col]
X = df.drop(columns=[target_col])

# Encode categorical variables (cp, restecg, thal - already numeric in UCI)
# Ensure all are numeric
for col in X.columns:
    if X[col].dtype == 'object' or X[col].dtype.name == 'category':
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Feature scaling for continuous numerical features (chol, trestbps, thalach)
scale_cols = ['chol', 'trestbps', 'thalach']
scale_cols = [c for c in scale_cols if c in X.columns]

scaler = StandardScaler()
X_scaled = X.copy()
if scale_cols:
    X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

# Data splitting 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Feature columns: {list(X.columns)}")

### Step 4: Model Building & Training with Hyperparameter Tuning

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Initialize models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

# GridSearchCV parameters for hyperparameter tuning
param_grids = {
    'Decision Tree': {'max_depth': [3, 5, 7, 10], 'min_samples_split': [2, 5, 10], 'criterion': ['gini', 'entropy']},
    'Random Forest': {'n_estimators': [100, 200], 'max_depth': [5, 10, 15], 'min_samples_split': [2, 5]},
    'Logistic Regression': {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']},
    'SVM': {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'], 'kernel': ['rbf', 'linear']}
}

best_models = {}

for name, model in models.items():
    print(f"\nTuning {name}...")
    grid = GridSearchCV(model, param_grids[name], cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    print(f"Best params: {grid.best_params_}")

### Step 5: Make Predictions & Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

results = []
predictions = {}

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    predictions[name] = (y_pred, y_prob)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else 0
    
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1, 'ROC-AUC': roc})

results_df = pd.DataFrame(results)
print("\nPerformance Summary (Best model by ROC-AUC and F1-Score):")
print(results_df.to_string(index=False))

In [ ]:
# Select best model based on ROC-AUC and F1-Score
results_df['Composite'] = results_df['ROC-AUC'] + results_df['F1-Score']
best_model_name = results_df.loc[results_df['Composite'].idxmax(), 'Model']
best_model = best_models[best_model_name]
print(f"\nBest Model: {best_model_name}")

In [ ]:
# Confusion Matrix for best model
y_pred_best, _ = predictions[best_model_name]
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print("\nConfusion Matrix:")
print("TN:", cm[0,0], "FP:", cm[0,1])
print("FN:", cm[1,0], "TP:", cm[1,1])

In [ ]:
# Feature Importance (for Decision Tree and Random Forest)
if best_model_name in ['Decision Tree', 'Random Forest']:
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances}).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp['Feature'], feat_imp['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    print(feat_imp)

### Step 6: Export Model & Scaler

In [ ]:
import pickle
import os

# For Google Colab: save to current directory, then zip and download
try:
    from google.colab import files
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    out_dir = '/content'
else:
    out_dir = os.path.join(os.getcwd(), '..', 'model')
    os.makedirs(out_dir, exist_ok=True)

# Save best model
with open(os.path.join(out_dir, 'heart_disease_model.pkl'), 'wb') as f:
    pickle.dump(best_model, f)
with open(os.path.join(out_dir, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
metadata = {'feature_columns': list(X.columns), 'scale_columns': scale_cols, 'best_model_name': best_model_name}
with open(os.path.join(out_dir, 'metadata.pkl'), 'wb') as f:
    pickle.dump(metadata, f)

print("Exported: heart_disease_model.pkl, scaler.pkl, metadata.pkl")
print(f"Best model: {best_model_name}")

if IN_COLAB:
    # Download all 3 files
    import shutil
    shutil.make_archive('model_files', 'zip', out_dir)
    files.download('model_files.zip')
    print("Download model_files.zip, extract, and put the 3 .pkl files in your project's model/ folder.")